# Exact match or string matching measure (using fuzzy logic)

In [8]:
from google.colab import drive;
drive.mount('/content/drive');
!pip install pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install -U scikit-learn


In [10]:
!pip install fuzzywuzzy pandas openpyxl

In [11]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/IST/Example/GPT_4_Turbo/GPCA/Result/GPT_4_Turbo_Result.xlsx'
df = pd.read_excel(file_path)
print(df.head())

     Run                                            ACAS XU  \
0  Run 1  G0: ACAS Xu satisfies security requirements\nG...   
1  Run 2  G0: "ACAS Xu satisfies security requirements"\...   
2  Run 3  G0: ACAS Xu satisfies security requirements\nG...   
3  Run 4  G0: ACAS Xu satisfies security requirements\nG...   
4  Run 5  G0: ACAS Xu satisfies security requirements\nG...   

                                            BlueROV2  \
0  G1: Hazards for "BlueROV2 System" addressed in...   
1  G1: Hazards for BlueROV2 System addressed in a...   
2  G1: Hazards for BlueROV2 system addressed in a...   
3  G1: Hazards for "BlueROV2 System" addressed in...   
4  G1: Hazards for the BlueROV2 system addressed ...   

                                            DeepMind  \
0  G1: Interpretability Claim. The DeepMind ML Mo...   
1  G1: Interpretability Claim. The DeepMind ML Mo...   
2  G1: Interpretability Claim Interpretability of...   
3  G1: Interpretability Claim. The DeepMind model...   
4  G

In [12]:
file_path2 = '/content/drive/MyDrive/Colab Notebooks/IST/Example/GPT_4_Turbo/GPCA/Result/Ground_Truth_AC.xlsx'
ground_truth_df = pd.read_excel(file_path2)
print(ground_truth_df.head())

          Case                                                 AC
0      ACAS XU  G0: ACAS Xu satisfies security requirements\nG...
1     BlueROV2  G1: Hazards for BlueROV2 addressed in accordan...
2     DeepMind  G1: Interpretability Claim - The ML system is ...
3  IM_Software  G1: IM server software is acceptable secure in...


In [13]:
from fuzzywuzzy import fuzz


for index, experiment_row in df.iterrows():
    for case in ['ACAS XU', 'BlueROV2', 'DeepMind', 'IM_Software']:
        experiment_text = experiment_row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]
        similarity_score = fuzz.ratio(experiment_text, ground_truth_text)/ 100.0

        print(f"Similarity score for {case} in {experiment_row['Run']}: {similarity_score}")
    print("-------")

Similarity score for ACAS XU in Run 1: 0.76
Similarity score for BlueROV2 in Run 1: 0.12
Similarity score for DeepMind in Run 1: 0.28
Similarity score for IM_Software in Run 1: 0.25
-------
Similarity score for ACAS XU in Run 2: 0.66
Similarity score for BlueROV2 in Run 2: 0.72
Similarity score for DeepMind in Run 2: 0.21
Similarity score for IM_Software in Run 2: 0.08
-------
Similarity score for ACAS XU in Run 3: 0.74
Similarity score for BlueROV2 in Run 3: 0.77
Similarity score for DeepMind in Run 3: 0.25
Similarity score for IM_Software in Run 3: 0.3
-------
Similarity score for ACAS XU in Run 4: 0.61
Similarity score for BlueROV2 in Run 4: 0.69
Similarity score for DeepMind in Run 4: 0.31
Similarity score for IM_Software in Run 4: 0.12
-------
Similarity score for ACAS XU in Run 5: 0.56
Similarity score for BlueROV2 in Run 5: 0.77
Similarity score for DeepMind in Run 5: 0.28
Similarity score for IM_Software in Run 5: 0.39
-------


# Bleu score

In [14]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 4.4 MB/s eta 0:00:00


In [15]:
import sacrebleu

def calculate_bleu_score(experiment_text, ground_truth_text):
    references = [[ground_truth_text]]
    system_output = [experiment_text]
    bleu = sacrebleu.corpus_bleu(system_output, references)
    # Scale the score to a range between 0 and 1 and round to two decimal points
    return round(bleu.score / 100, 2)

for case in ['ACAS XU', 'BlueROV2', 'DeepMind', 'IM_Software']:
    print(f"Calculating BLEU scores for {case}:")
    for index, row in df.iterrows():
        experiment_text = row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

        if pd.notnull(experiment_text) and pd.notnull(ground_truth_text):
            bleu_score = calculate_bleu_score(experiment_text, ground_truth_text)
            print(f"Run {index + 1} BLEU score: {bleu_score}")
        else:
            print(f"Run {index + 1} BLEU score: Data Missing")
    print("-------")


Calculating BLEU scores for ACAS XU:
Run 1 BLEU score: 0.81
Run 2 BLEU score: 0.48
Run 3 BLEU score: 0.75
Run 4 BLEU score: 0.69
Run 5 BLEU score: 0.59
-------
Calculating BLEU scores for BlueROV2:
Run 1 BLEU score: 0.33
Run 2 BLEU score: 0.72
Run 3 BLEU score: 0.63
Run 4 BLEU score: 0.56
Run 5 BLEU score: 0.68
-------
Calculating BLEU scores for DeepMind:
Run 1 BLEU score: 0.2
Run 2 BLEU score: 0.23
Run 3 BLEU score: 0.13
Run 4 BLEU score: 0.2
Run 5 BLEU score: 0.2
-------
Calculating BLEU scores for IM_Software:
Run 1 BLEU score: 0.3
Run 2 BLEU score: 0.3
Run 3 BLEU score: 0.33
Run 4 BLEU score: 0.33
Run 5 BLEU score: 0.31
-------


# Semantic Similarity

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

for case in ['ACAS XU', 'BlueROV2', 'DeepMind', 'IM_Software']:
    experiment_texts = df[case].tolist()
    ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

    combined_texts = experiment_texts + [ground_truth_text]

    tfidf_matrix = vectorizer.fit_transform(combined_texts)

    experiment_vectors = tfidf_matrix[:-1]
    ground_truth_vector = tfidf_matrix[-1:]

    cosine_similarities = cosine_similarity(experiment_vectors, ground_truth_vector)

    for i, similarity_score in enumerate(cosine_similarities.flatten(), 1):
        # Round the score to two decimal places without multiplying by 100
        rounded_score = round(similarity_score, 2)
        print(f"Cosine similarity for {case}, Run {i}: {rounded_score}")
    print("-------")


Cosine similarity for ACAS XU, Run 1: 0.91
Cosine similarity for ACAS XU, Run 2: 0.92
Cosine similarity for ACAS XU, Run 3: 0.9
Cosine similarity for ACAS XU, Run 4: 0.89
Cosine similarity for ACAS XU, Run 5: 0.85
-------
Cosine similarity for BlueROV2, Run 1: 0.85
Cosine similarity for BlueROV2, Run 2: 0.91
Cosine similarity for BlueROV2, Run 3: 0.84
Cosine similarity for BlueROV2, Run 4: 0.82
Cosine similarity for BlueROV2, Run 5: 0.85
-------
Cosine similarity for DeepMind, Run 1: 0.55
Cosine similarity for DeepMind, Run 2: 0.57
Cosine similarity for DeepMind, Run 3: 0.45
Cosine similarity for DeepMind, Run 4: 0.51
Cosine similarity for DeepMind, Run 5: 0.55
-------
Cosine similarity for IM_Software, Run 1: 0.66
Cosine similarity for IM_Software, Run 2: 0.7
Cosine similarity for IM_Software, Run 3: 0.71
Cosine similarity for IM_Software, Run 4: 0.72
Cosine similarity for IM_Software, Run 5: 0.72
-------
